In [1]:
import cv2
import os
from pathlib import Path
from tqdm.notebook import tqdm

# Mapping label untuk N-RDD2024
label_mapping = {
    'D00': '0', 'D10': '1', 'D20': '2', 'D30': '3', 'D40': '4', 
    'D50': '5', 'D60': '6', 'D70': '7', 'D80': '8', 'D90': '9'
}

# Sesuaikan dengan lokasi dataset Anda
dataset_dir = Path('/home/sabda/code/pothole-detection/dataset')

print("[*] Memulai proses pembersihan label hybrid...")

for split in ['train', 'val']:
    labels_path = dataset_dir / split / 'labels'
    images_path = dataset_dir / split / 'images'
    
    if not labels_path.exists(): continue
    
    files = list(labels_path.glob('*.txt'))
    for txt_file in tqdm(files, desc=f"Memproses {split}"):
        # Cari gambar untuk mendapatkan dimensi (penting untuk normalisasi)
        img_exts = ['.jpg', '.jpeg', '.png']
        img_file = None
        for ext in img_exts:
            if (images_path / (txt_file.stem + ext)).exists():
                img_file = images_path / (txt_file.stem + ext)
                break
        
        if img_file is None: continue
        
        img = cv2.imread(str(img_file))
        h, w = img.shape[:2]
        
        with open(txt_file, 'r') as f:
            lines = f.readlines()
            
        new_lines = []
        modified = False
        
        for line in lines:
            parts = line.strip().split()
            if not parts: continue
            
            # KASUS 1: Format sudah benar (5 kolom: ID x y w h)
            if len(parts) == 5 and parts[0].isdigit():
                new_lines.append(line)
            
            # KASUS 2: Format Polygon (9-10 kolom: x1 y1 x2 y2 x3 y3 x4 y4 Label Flag)
            elif len(parts) >= 9:
                try:
                    coords = [float(p) for p in parts[:8]]
                    raw_label = parts[8]
                    class_id = label_mapping.get(raw_label, "0") # Default ke 0 jika tidak ada di map
                    
                    xs = coords[0::2]
                    ys = coords[1::2]
                    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
                    
                    # Konversi ke YOLO Normalized
                    x_center = ((xmin + xmax) / 2) / w
                    y_center = ((ymin + ymax) / 2) / h
                    bw = (xmax - xmin) / w
                    bh = (ymax - ymin) / h
                    
                    new_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}\n")
                    modified = True
                except:
                    continue
                    
        if modified:
            with open(txt_file, 'w') as f:
                f.writelines(new_lines)

print("\n[*] Selesai! Dataset sekarang seragam dalam format YOLO.")

[*] Memulai proses pembersihan label hybrid...


Memproses train:   0%|          | 0/19306 [00:00<?, ?it/s]

Memproses val:   0%|          | 0/4127 [00:00<?, ?it/s]


[*] Selesai! Dataset sekarang seragam dalam format YOLO.
